In [136]:
import numpy as np
import pandas as pd
import yfinance as yf
import requests

In [142]:
# Gets list of crypto tickers from coingecko and formats them into a pandas dataframe
crypto_request = requests.get("https://api.coingecko.com/api/v3/coins/markets",
                 params={"vs_currency": "usd", "per_page": 250, "order": "market_cap_desc"})
crypto_dataframe = pd.DataFrame(crypto_request.json()).iloc[:30]
crypto_dataframe[["id", "symbol", "name", "current_price", "market_cap", "total_volume"]]

,id,symbol,name,current_price,market_cap,total_volume
0,bitcoin,btc,Bitcoin,76265.000000,1531963677085,3.470149e+10
1,ethereum,eth,Ethereum,2417.260000,295130453701,1.006580e+19
2,tether,usdt,Tether,0.999511,183335273084,6.117494e+10
3,binancecoin,bnb,BNB,717.830000,95615032663,9.155167e+08
4,ripple,xrp,XRP,1.380000,87057734365,4.436822e+09
5,usd-coin,usdc,USDC,0.999742,73978680186,1.831188e+10
6,solana,sol,Solana,98.830000,58059127681,3.375999e+09
7,tron,trx,TRON,0.335123,31829106223,4.638111e+08
8,figure-heloc,figr_heloc,Figure Heloc,1.046000,23658656113,5.598223e+07
9,zcash,zec,Zcash,1114.320000,18872543905,9.036114e+08


In [143]:
# Reformats the ticker symbols and reformats them into COIN-USD form to be read by yahoo finance
ticker_symbols = crypto_dataframe["symbol"].map(lambda x: np.char.upper(x) + "-USD").values.astype(str).tolist()
ticker_symbols

['BTC-USD',
 'ETH-USD',
 'USDT-USD',
 'BNB-USD',
 'XRP-USD',
 'USDC-USD',
 'SOL-USD',
 'TRX-USD',
 'FIGR_HELOC-USD',
 'ZEC-USD',
 'HYPE-USD',
 'DOGE-USD',
 'USDS-USD',
 'RAIN-USD',
 'XMR-USD',
 'WBT-USD',
 'LINK-USD',
 'LEO-USD',
 'ADA-USD',
 'XLM-USD',
 'USDE-USD',
 'DAI-USD',
 'BCH-USD',
 'USD1-USD',
 'LTC-USD',
 'UNI-USD',
 'GRAM-USD',
 'CC-USD',
 'HBAR-USD',
 'USDG-USD']

In [144]:
# Enters each ticker into Yahoo finance to check if it is valid
def attemptDownloads(tickers):
    valid_tickers = []
    for ticker in tickers:
        dataframe = yf.download(tickers=ticker, period="max", interval="1d")
        if len(dataframe) > 0:
            valid_tickers.append(ticker)
    return valid_tickers
valid_tickers = attemptDownloads(ticker_symbols)
valid_tickers

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
$FIGR_HELOC-USD: possibly delisted; no timezone found
[*********************100%***********************]  1 of 1 completed

1 Failed download:
['FIGR_HELOC-USD']: possibly delisted; no timezone found
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***************

['BTC-USD',
 'ETH-USD',
 'USDT-USD',
 'BNB-USD',
 'XRP-USD',
 'USDC-USD',
 'SOL-USD',
 'TRX-USD',
 'ZEC-USD',
 'HYPE-USD',
 'DOGE-USD',
 'USDS-USD',
 'RAIN-USD',
 'XMR-USD',
 'WBT-USD',
 'LINK-USD',
 'LEO-USD',
 'ADA-USD',
 'XLM-USD',
 'USDE-USD',
 'DAI-USD',
 'BCH-USD',
 'USD1-USD',
 'LTC-USD',
 'UNI-USD',
 'GRAM-USD',
 'CC-USD',
 'HBAR-USD',
 'USDG-USD']

In [160]:
ticker_universe = ["BTC-USD", "ETH-USD", "LINK-USD", "SOL-USD", "AVAX-USD", "XRP-USD", "BNB-USD", "LTC-USD", "ADA-USD", "DOT-USD", "DOGE-USD", "MATIC-USD"]

# ticker_universe = valid_tickers
# len(ticker_universe)

In [161]:
# Downloads the prices from yfinance
daily_prices = yf.download(tickers=ticker_universe, period="max", interval="1d")

[*********************100%***********************]  12 of 12 completed


In [162]:
# Calculates daily returns and daily volumes from daily prices
daily_returns = (daily_prices["Close"] / daily_prices["Close"].shift() - 1).iloc[1:].loc["2020-01-01":]
daily_volumes = daily_prices["Volume"].loc["2020-01-01":]

In [163]:
def zScore(x, window=120):
    return (x - x.rolling(window).mean()) / x.rolling(window).std()

In [164]:
def sharpeRatio(x):
    return x.mean() / x.std() * np.sqrt(252)

In [165]:
# Calculates weights for momentum strategy
# rolling periods for signal calculations can be varied
def momentumWeights(returns, returns_rolling=7, volatility_rolling=60):
    signal = zScore(returns.shift(), window=120).rolling(returns_rolling).mean() / returns.shift().rolling(volatility_rolling).std()
    ranked = signal.rank(axis=1)
    demeaned = ranked.sub(ranked.mean(axis=1), axis=0)
    weights = demeaned.div(demeaned.abs().sum(axis=1), axis=0)
    return weights

In [166]:
# Reversal strategy (still a work in progress)
def reversalWeights(returns, volumes, returns_rolling=7, volatility_rolling=720):
    signal = (returns.shift().rolling(returns_rolling).mean() / (volumes / volumes.rolling(volatility_rolling).mean())).dropna()
    ranked = signal.rank(axis=1)
    demeaned = ranked.sub(ranked.mean(axis=1), axis=0)
    normalised = demeaned.div(demeaned.abs().sum(axis=1), axis=0)
    return -normalised

In [167]:
# Creating the returns for the momentum strategy
momentum_strategy = (momentumWeights(daily_returns) * daily_returns).sum(axis=1)
reversal_strategy = (reversalWeights(daily_returns, daily_volumes) * daily_returns).sum(axis=1)

In [168]:
# Calculates important summary metrics for the strategy
def strategySummary(strategy, rolling_window=14, market_threshold=0.08):
    market_trend = daily_returns['BTC-USD'].shift().rolling(window=rolling_window).sum()

    bullish_trend = market_trend > market_threshold
    bearish_trend = market_trend < -market_threshold
    choppy_trend = (market_trend > -market_threshold) & (market_trend < market_threshold)

    print(f"Overall Sharpe Ratio: {sharpeRatio(strategy):.3f}")
    print("")
    print(f"Choppy market Sharpe: {sharpeRatio(strategy[choppy_trend]):.3f}")
    print(f"Bullish market Sharpe: {sharpeRatio(strategy[bullish_trend]):.3f}")
    print(f"Bearish market Sharpe: {sharpeRatio(strategy[bearish_trend]):.3f}")
    print()
    print(f"Hit Rate: {(strategy > 0).mean():.3f}")
    print(f"Avg Win: {strategy[strategy > 0].mean():.3f}")
    print(f"Avg Loss: {strategy[strategy < 0].mean():.3f}")
    print(f"Win/Loss Ratio: {-strategy[strategy > 0].mean() / strategy[strategy < 0].mean():.2f}")
    print("")
    print(f"Percentage of choppy days: {choppy_trend.mean():.2f}")

In [169]:
strategySummary(momentum_strategy)

Overall Sharpe Ratio: 1.433

Choppy market Sharpe: 0.917
Bullish market Sharpe: 2.103
Bearish market Sharpe: 1.975

Hit Rate: 0.489
Avg Win: 0.009
Avg Loss: -0.007
Win/Loss Ratio: 1.27

Percentage of choppy days: 0.60
